In [1]:
import os
from pathlib import Path
import gradio as gr
import json
import re
from typing import Dict, List

# # 读取数据
# with open('pred_traj/toolalpaca/gpt-3.5-turbo/graph_eval_two_shot.json', 'r', encoding='utf-8') as f:
#     data = json.load(f)

# 读取所有JSON文件并合并数据
def load_all_json_files(root_dir: str) -> List[Dict]:
    all_data = []
    root_path = Path(root_dir)
    
    # 递归遍历所有子目录
    for json_file in root_path.rglob('*.json'):
        try:
            with open(json_file, 'r', encoding='utf-8') as f:
                file_data = json.load(f)
                if isinstance(file_data, list):
                    all_data.extend(file_data)
                else:
                    print(f"警告: {json_file} 不是列表格式")
        except Exception as e:
            print(f"读取 {json_file} 时出错: {str(e)}")
    
    return all_data

# 替换原来的数据读取代码
data = load_all_json_files('pred_traj')

# 按source分组数据
def group_by_source(data: List[Dict]) -> Dict[str, List[Dict]]:
    grouped_data = {}
    for item in data:
        source = item['query']['source']
        if source not in grouped_data:
            grouped_data[source] = []
        grouped_data[source].append(item)
    return grouped_data

grouped_data = group_by_source(data)

def display_item(item: Dict, index: int, total: int) -> str:
    """格式化显示单个数据项"""
    conversations = item['query']['conversations']

    formatted_text = ""
    
    for conv in conversations:
        role = conv['role']
        # 处理content中的特殊格式
        content = conv['content']
        
        # 如果内容包含代码块或特殊格式，使用代码块格式显示
        if 'Node:' in content or 'Edge:' in content:
            content = f"```\n{content}\n```"
        else:
            # 普通文本的换行处理
            try:    
                api_list = content.split('\n')[-1][8:].strip(" :")
                content = '\n'.join(content.split('\n')[:-1])
                content = content.replace('\n', '<br><br>')
                content += f"<br>API List: \n```json\n{api_list}\n```"
            except:
                content = content.replace('\n', '<br><br>')
        # content = f"```\n{content}\n```"
        
        # 角色显示美化
        role_display = {
            'system': '🤖 系统',
            'user': '👤 用户',
            'assistant': '🤖 助手（gold）'
        }.get(role, role)
        
        formatted_text += f"### {role_display}\n{content}\n\n"
    
    # 如果有workflow字段，也显示出来
    if 'workflow' in item:
        formatted_text += "### 📋 工作流\n"
        formatted_text += f"```\n{item['workflow']}\n```\n\n"

    
    # 添加分隔线使显示更清晰
    formatted_text += "---\n\n"
    formatted_text += f"## 示例 {index + 1}/{total}\n\n"
    
    return formatted_text


def create_interface():
    with gr.Blocks(theme=gr.themes.Soft()) as demo:
        gr.Markdown("# 对话数据查看器")
        
        with gr.Tabs() as tabs:
            # 为每个source创建一个选项卡
            tab_states = {}
            for source in grouped_data.keys():
                with gr.Tab(source):
                    
                    # 显示当前数据
                    markdown_output = gr.Markdown()
                    
                    # 存储当前索引的状态
                    current_index = gr.State(value=0)

                    with gr.Row():
                        prev_btn = gr.Button("上一条", variant="secondary")
                        next_btn = gr.Button("下一条", variant="secondary")
                    
                    # 更新显示函数
                    def update_display(index, source=source):
                        items = grouped_data[source]
                        return display_item(items[index], index, len(items))
                    
                    # 上一条按钮点击事件
                    def prev_item(index, source=source):
                        items = grouped_data[source]
                        new_index = (index - 1) % len(items)
                        return new_index, update_display(new_index, source)
                    
                    # 下一条按钮点击事件
                    def next_item(index, source=source):
                        items = grouped_data[source]
                        new_index = (index + 1) % len(items)
                        return new_index, update_display(new_index, source)
                    
                    # 设置初始显示
                    markdown_output.value = update_display(0, source)
                    
                    # 绑定按钮事件
                    prev_btn.click(
                        prev_item,
                        inputs=[current_index],
                        outputs=[current_index, markdown_output]
                    )
                    
                    next_btn.click(
                        next_item,
                        inputs=[current_index],
                        outputs=[current_index, markdown_output]
                    )
                    
                    tab_states[source] = {
                        'markdown': markdown_output,
                        'current_index': current_index
                    }

    return demo

# 启动应用
demo = create_interface()
demo.launch(server_port=6006)

/root/miniconda3/envs/WorFBench/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


* Running on local URL:  http://127.0.0.1:6006

To create a public link, set `share=True` in `launch()`.


In [1]:
from LLM.localLLM import localLLM
from LLM.openaiLLM import openaiLLM

def test_workflow_generation():
    # 构造测试消息
    test_messages = [
        {
            "role": "system",
            "content": "你是一个工作流程规划助手。请帮助用户将任务分解为具体的步骤。"
        },
        {
            "role": "user", 
            "content": "请告诉我如何制作一杯咖啡的步骤"
        }
    ]
    
    try:
        # 调用localLLM获取响应
        print(test_messages)
        # response = localLLM(messages=test_messages, api_port=8000)
        response = openaiLLM(messages=test_messages)
        
        print("生成的工作流:")
        print(response)
        
        
        print("测试通过!")
        return True
        
    except Exception as e:
        print(f"测试失败: {str(e)}")
        return False

if __name__ == "__main__":
    test_workflow_generation()

[{'role': 'system', 'content': '你是一个工作流程规划助手。请帮助用户将任务分解为具体的步骤。'}, {'role': 'user', 'content': '请告诉我如何制作一杯咖啡的步骤'}]
生成的工作流:
{'error': 'Connection error.'}
测试通过!
